
# Time-Series Classification with MLP
## UCI Human Activity Recognition (HAR)

### 실습 목표
이번 실습에서는 스마트폰의 **가속도계(accelerometer)** 및 **자이로스코프(gyroscope)** 시계열 데이터를 사용하여 사람의 행동을 분류합니다.

사용할 모델은 **MLP (Multi-Layer Perceptron)** 로 제한합니다.  

가장 높은 **test 성능**을 내는 것을 목적으로 합니다.

참가자들은 **validation 성능**을 기준으로 모델 구조와 하이퍼파라미터를 탐색하고, 최종적으로 가장 좋은 설정을 선택합니다.

### 분류 대상
1. WALKING
2. WALKING_UPSTAIRS
3. WALKING_DOWNSTAIRS
4. SITTING
5. STANDING
6. LAYING

### 핵심 규칙
- CNN, RNN, LSTM, Transformer는 사용하지 않습니다.
- 입력 시계열 `(128, 9)`를 **flatten**하여 MLP에 입력합니다.
- 모델 선택은 **validation accuracy**를 기준으로 합니다.
- **test set은 최종 모델을 선택한 뒤 마지막에 평가**합니다.
- validation split은 가능한 한 subject가 겹치지 않도록 구성합니다.

> UCI HAR Dataset: Human Activity Recognition Using Smartphones  
> 각 sample은 50 Hz로 측정된 2.56초 길이의 window이며, 총 128 time steps로 구성됩니다.


In [ ]:

# 기본 라이브러리
import os
import random
import zipfile
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())



## 1. Reproducibility 설정

동일한 실험을 반복했을 때 결과가 크게 달라지지 않도록 random seed를 고정합니다.


In [ ]:

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)



## 2. UCI HAR Dataset 다운로드

원본 데이터의 `Inertial Signals`를 사용합니다.

9개의 시계열 채널:
- body acceleration: x, y, z
- body gyroscope: x, y, z
- total acceleration: x, y, z

각 sample의 shape:

\[
128 	imes 9
\]

MLP에서는 이를 flatten하여 1,152차원 벡터로 사용합니다.


In [ ]:
DATA_DIR = Path("./uci_har")
ZIP_PATH = DATA_DIR / "UCI HAR Dataset.zip" # Use the filename with spaces to match the URL

DATA_DIR.mkdir(exist_ok=True)

# UCI 공식 파일 주소
urls = [
    "https://archive.ics.uci.edu/static/public/240/human+activity+recognition+using+smartphones.zip",
    "https://archive.ics.uci.edu/ml/machine-learning-databases/00240/UCI%20HAR%20Dataset.zip",
]

def find_dataset_root(base_data_dir: Path) -> Path | None:
    """
    Attempts to find the actual root directory containing 'train' and 'test' folders.
    Checks for direct extraction, single nesting, and double nesting.
    """
    # Option 1: Dataset contents extracted directly into base_data_dir
    if (base_data_dir / "train").is_dir() and (base_data_dir / "test").is_dir():
        return base_data_dir

    # Option 2: Dataset contents are in a single subdirectory within base_data_dir
    # e.g., base_data_dir / "UCI HAR Dataset" / train/
    for item in base_data_dir.iterdir():
        if item.is_dir() and (item / "train").is_dir() and (item / "test").is_dir():
            return item

    # Option 3: Double nesting, e.g., base_data_dir / "Folder" / "SubFolder" / train/
    for item in base_data_dir.iterdir():
        if item.is_dir():
            for sub_item in item.iterdir():
                if sub_item.is_dir() and (sub_item / "train").is_dir() and (sub_item / "test").is_dir():
                    return sub_item
    return None

# Attempt to find if dataset is already extracted in a recognizable structure
actual_extract_dir = find_dataset_root(DATA_DIR)

if actual_extract_dir is None:
    # Proceed with download and extraction if not found
    # Ensure any previous inconsistent zip file is removed
    if ZIP_PATH.exists():
        print(f"Removing existing zip file: {ZIP_PATH}")
        ZIP_PATH.unlink()

    # Robust download loop: try all URLs until a valid zip is obtained AND successfully extracted
    downloaded_extracted_and_valid = False
    for url in urls:
        print(f"Attempting download from: {url}")
        try:
            urllib.request.urlretrieve(url, ZIP_PATH)
            if zipfile.is_zipfile(ZIP_PATH):
                print(f"Successfully downloaded and verified zip from: {url}")
                print("Extracting...")
                with zipfile.ZipFile(ZIP_PATH, "r") as z:
                    z.extractall(DATA_DIR) # Extracts to ./uci_har
                # If extraction completes without error, it's good
                downloaded_extracted_and_valid = True
                break
            else:
                print(f"Downloaded file from {url} is not a valid zip archive. Removing and trying next URL.")
                # Remove the invalid zip file before trying next URL
                if ZIP_PATH.exists():
                    ZIP_PATH.unlink()
        except Exception as e:
            print(f"Download from {url} or extraction failed: {e}. Removing potential partial file and trying next URL.")
            # Ensure any partial download or problematic zip is removed
            if ZIP_PATH.exists():
                ZIP_PATH.unlink()
            # Also remove any partially extracted files if DATA_DIR is not empty (careful not to delete user files)
            # For this context, assuming DATA_DIR only contains this dataset, we can clear it.
            # Alternatively, we could be more surgical, but for now, this is simpler.
            for item in DATA_DIR.iterdir():
                if item != ZIP_PATH: # Don't remove the zip we're about to remove anyway
                    if item.is_dir():
                        shutil.rmtree(item)
                    else:
                        item.unlink()

    if not downloaded_extracted_and_valid:
        raise RuntimeError(
            "Dataset download and extraction failed from all sources, or extracted files were corrupted. "
            "Please download 'UCI HAR Dataset.zip' manually from the UCI repository "
            "and place it at ./uci_har/UCI HAR Dataset.zip"
        )

    # After successful extraction, try to find the root again
    actual_extract_dir = find_dataset_root(DATA_DIR)

    if actual_extract_dir is None:
        # If still not found after extraction (should not happen if downloaded_extracted_and_valid is True), raise an informative error
        extracted_items = [str(p) for p in DATA_DIR.iterdir()]
        raise RuntimeError(f"Could not find 'train' or 'test' directories after extraction in {DATA_DIR}. "
                             f"Contents of {DATA_DIR}: {extracted_items}. "
                             "Please check the zip file's internal structure or extract manually.")
else:
    print(f"Dataset already extracted to: {actual_extract_dir}")

EXTRACT_DIR = actual_extract_dir # Set the global EXTRACT_DIR for subsequent cells

print("Dataset path:", EXTRACT_DIR)


## 3. Raw time-series signal 불러오기

공식 train/test split은 **subject 기준**으로 나뉘어 있습니다.

여기서는:
- Official Train → 다시 **Train / Validation**으로 분리
- Official Test → 마지막 최종 평가에만 사용

validation도 subject 기준으로 분리하여 같은 사람이 train과 validation에 동시에 들어가는 것을 방지합니다.


In [ ]:

SIGNAL_NAMES = [
    "body_acc_x",
    "body_acc_y",
    "body_acc_z",
    "body_gyro_x",
    "body_gyro_y",
    "body_gyro_z",
    "total_acc_x",
    "total_acc_y",
    "total_acc_z",
]

def load_signals(split):
    signal_dir = EXTRACT_DIR / split / "Inertial Signals"
    signals = []

    for name in SIGNAL_NAMES:
        path = signal_dir / f"{name}_{split}.txt"
        x = np.loadtxt(path)
        signals.append(x)

    # list of (N, 128) -> (N, 128, 9)
    return np.stack(signals, axis=-1)

def load_labels(split):
    y = np.loadtxt(EXTRACT_DIR / split / f"y_{split}.txt").astype(int)
    return y - 1  # 1~6 -> 0~5

def load_subjects(split):
    return np.loadtxt(
        EXTRACT_DIR / split / f"subject_{split}.txt"
    ).astype(int)

X_official_train = load_signals("train")
y_official_train = load_labels("train")
subject_official_train = load_subjects("train")

X_test = load_signals("test")
y_test = load_labels("test")
subject_test = load_subjects("test")

print("Official Train:", X_official_train.shape, y_official_train.shape)
print("Official Test :", X_test.shape, y_test.shape)
print("Number of channels:", len(SIGNAL_NAMES))


In [ ]:

CLASS_NAMES = [
    "WALKING",
    "WALKING_UPSTAIRS",
    "WALKING_DOWNSTAIRS",
    "SITTING",
    "STANDING",
    "LAYING",
]

print("Training subjects:", np.unique(subject_official_train))
print("Test subjects    :", np.unique(subject_test))

train_counts = pd.Series(y_official_train).value_counts().sort_index()
display(pd.DataFrame({
    "class_id": range(6),
    "class_name": CLASS_NAMES,
    "count": train_counts.values
}))



## 4. Train / Validation split

**중요:** 일반적인 random split을 하면 같은 subject의 window가 train과 validation에 동시에 포함될 수 있습니다.

여기서는 `GroupShuffleSplit`을 사용하여 **subject 단위**로 validation set을 만듭니다.


In [ ]:

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=SEED,
)

train_idx, val_idx = next(
    splitter.split(
        X_official_train,
        y_official_train,
        groups=subject_official_train,
    )
)

X_train = X_official_train[train_idx]
y_train = y_official_train[train_idx]

X_val = X_official_train[val_idx]
y_val = y_official_train[val_idx]

train_subjects = np.unique(subject_official_train[train_idx])
val_subjects = np.unique(subject_official_train[val_idx])

print("Train shape:", X_train.shape)
print("Val shape  :", X_val.shape)
print("Test shape :", X_test.shape)

print("\nTrain subjects:", train_subjects)
print("Val subjects  :", val_subjects)
print("Overlap       :", np.intersect1d(train_subjects, val_subjects))



## 5. 입력 정규화

각 sensor channel에 대해 **training set만 사용하여** 평균과 표준편차를 계산합니다.

Validation 및 Test에는 training statistics를 그대로 적용합니다.

이렇게 해야 data leakage를 피할 수 있습니다.


In [ ]:

# X_train shape: (N, 128, 9)
channel_mean = X_train.mean(axis=(0, 1), keepdims=True)
channel_std = X_train.std(axis=(0, 1), keepdims=True) + 1e-8

def normalize(X):
    return (X - channel_mean) / channel_std

X_train_norm = normalize(X_train)
X_val_norm = normalize(X_val)
X_test_norm = normalize(X_test)

print("Mean after normalization:")
print(X_train_norm.mean(axis=(0, 1)))

print("\nStd after normalization:")
print(X_train_norm.std(axis=(0, 1)))



## 6. 시계열 데이터 확인

아래에서는 한 sample의 9개 sensor signal 중 일부를 시각화합니다.


In [ ]:

sample_idx = 0

plt.figure(figsize=(12, 5))
for ch in range(3):
    plt.plot(
        X_train_norm[sample_idx, :, ch],
        label=SIGNAL_NAMES[ch],
    )

plt.xlabel("Time step")
plt.ylabel("Normalized value")
plt.title(f"Example sample: {CLASS_NAMES[y_train[sample_idx]]}")
plt.legend()
plt.show()



## 7. PyTorch DataLoader 만들기

MLP 입력을 위해 `(128, 9)`를 flatten합니다.

\[
128 	imes 9 = 1152
\]


In [ ]:

def make_loader(X, y, batch_size=128, shuffle=False):
    X_flat = X.reshape(len(X), -1).astype(np.float32)
    y = y.astype(np.int64)

    dataset = TensorDataset(
        torch.tensor(X_flat),
        torch.tensor(y),
    )

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
    )

BATCH_SIZE = 128

train_loader = make_loader(
    X_train_norm, y_train,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

val_loader = make_loader(
    X_val_norm, y_val,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

test_loader = make_loader(
    X_test_norm, y_test,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

INPUT_DIM = 128 * 9
NUM_CLASSES = 6

print("Input dimension:", INPUT_DIM)



# 8. Baseline MLP

먼저 매우 단순한 baseline을 학습합니다.

Baseline:

\[
1152
ightarrow 128
ightarrow 6
\]

이후 학생들은 이 baseline보다 높은 validation accuracy를 얻도록 모델을 개선합니다.


In [ ]:

class MLP(nn.Module):
    def __init__(
        self,
        input_dim,
        hidden_dims,
        num_classes,
        activation="relu",
        dropout=0.0,
        batch_norm=False,
    ):
        super().__init__()

        activation_dict = {
            "relu": nn.ReLU,
            "gelu": nn.GELU,
            "silu": nn.SiLU,
            "leaky_relu": nn.LeakyReLU,
            "tanh": nn.Tanh,
        }

        if activation not in activation_dict:
            raise ValueError(f"Unknown activation: {activation}")

        layers = []
        prev_dim = input_dim

        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, hidden_dim))

            if batch_norm:
                layers.append(nn.BatchNorm1d(hidden_dim))

            layers.append(activation_dict[activation]())

            if dropout > 0:
                layers.append(nn.Dropout(dropout))

            prev_dim = hidden_dim

        layers.append(nn.Linear(prev_dim, num_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


In [ ]:

baseline_model = MLP(
    input_dim=INPUT_DIM,
    hidden_dims=[128],
    num_classes=NUM_CLASSES,
    activation="relu",
    dropout=0.0,
    batch_norm=False,
).to(device)

print(baseline_model)

num_params = sum(p.numel() for p in baseline_model.parameters())
print(f"Number of parameters: {num_params:,}")



## 9. Training / Evaluation 함수

매 epoch마다 training loss와 validation accuracy를 기록합니다.

**모델 선택은 validation accuracy 기준**으로 수행합니다.


In [ ]:

def evaluate(model, loader):
    model.eval()

    all_preds = []
    all_targets = []
    total_loss = 0.0
    criterion = nn.CrossEntropyLoss()

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            logits = model(X_batch)
            loss = criterion(logits, y_batch)

            total_loss += loss.item() * X_batch.size(0)

            preds = logits.argmax(dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(y_batch.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    accuracy = accuracy_score(all_targets, all_preds)

    return avg_loss, accuracy, np.array(all_preds), np.array(all_targets)


def train_model(
    model,
    train_loader,
    val_loader,
    epochs=30,
    lr=1e-3,
    weight_decay=0.0,
    optimizer_name="adam",
    verbose=True,
):
    criterion = nn.CrossEntropyLoss()

    if optimizer_name.lower() == "adam":
        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=lr,
            weight_decay=weight_decay,
        )
    elif optimizer_name.lower() == "adamw":
        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=lr,
            weight_decay=weight_decay,
        )
    elif optimizer_name.lower() == "sgd":
        optimizer = torch.optim.SGD(
            model.parameters(),
            lr=lr,
            momentum=0.9,
            weight_decay=weight_decay,
        )
    else:
        raise ValueError(f"Unknown optimizer: {optimizer_name}")

    history = {
        "train_loss": [],
        "val_loss": [],
        "val_acc": [],
    }

    best_val_acc = -1.0
    best_state = None
    best_epoch = -1

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0

        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()

            logits = model(X_batch)
            loss = criterion(logits, y_batch)

            loss.backward()
            optimizer.step()

            running_loss += loss.item() * X_batch.size(0)

        train_loss = running_loss / len(train_loader.dataset)
        val_loss, val_acc, _, _ = evaluate(model, val_loader)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = epoch
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }

        if verbose:
            print(
                f"Epoch {epoch:02d} | "
                f"train loss: {train_loss:.4f} | "
                f"val loss: {val_loss:.4f} | "
                f"val acc: {val_acc:.4f}"
            )

    model.load_state_dict(best_state)

    return model, history, best_val_acc, best_epoch



## 10. Baseline 학습


In [ ]:

baseline_model = MLP(
    input_dim=INPUT_DIM,
    hidden_dims=[128],
    num_classes=NUM_CLASSES,
    activation="relu",
    dropout=0.0,
    batch_norm=False,
).to(device)

baseline_model, baseline_history, baseline_best_acc, baseline_best_epoch = train_model(
    baseline_model,
    train_loader,
    val_loader,
    epochs=30,
    lr=1e-3,
    weight_decay=0.0,
    optimizer_name="adam",
)

print()
print(f"Best validation accuracy: {baseline_best_acc:.4f}")
print(f"Best epoch: {baseline_best_epoch}")


In [ ]:

plt.figure(figsize=(8, 4))
plt.plot(baseline_history["val_acc"])
plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy")
plt.title("Baseline MLP")
plt.show()



# 11. 실습: 나만의 MLP 만들기

이제 아래 hyperparameter들을 자유롭게 변경해 봅니다.

### 탐색할 수 있는 요소
- `hidden_dims`
- `activation`
- `dropout`
- `batch_norm`
- `learning_rate`
- `weight_decay`
- `optimizer`
- `batch_size`
- `epochs`

### 예시 search space

| Hyperparameter | Example |
|---|---|
| # hidden layers | 1 ~ 5 |
| hidden dimension | 32, 64, 128, 256, 512, 1024 |
| activation | ReLU, GELU, SiLU, LeakyReLU, Tanh |
| dropout | 0.0 ~ 0.5 |
| batch normalization | True / False |
| learning rate | 1e-5 ~ 1e-2 |
| weight decay | 0 ~ 1e-2 |
| optimizer | SGD, Adam, AdamW |
| batch size | 32, 64, 128, 256 |

> **목표:** Baseline보다 높은 validation accuracy를 얻으세요.


In [ ]:

# ============================================================
# TODO: 아래 설정을 자유롭게 변경하세요.
# ============================================================

MY_CONFIG = {
    "hidden_dims": [256, 128],
    "activation": "relu",
    "dropout": 0.2,
    "batch_norm": False,

    "optimizer": "adam",
    "learning_rate": 1e-3,
    "weight_decay": 0.0,

    "batch_size": 128,
    "epochs": 30,
}

MY_CONFIG


In [ ]:

my_train_loader = make_loader(
    X_train_norm,
    y_train,
    batch_size=MY_CONFIG["batch_size"],
    shuffle=True,
)

my_val_loader = make_loader(
    X_val_norm,
    y_val,
    batch_size=MY_CONFIG["batch_size"],
    shuffle=False,
)

my_model = MLP(
    input_dim=INPUT_DIM,
    hidden_dims=MY_CONFIG["hidden_dims"],
    num_classes=NUM_CLASSES,
    activation=MY_CONFIG["activation"],
    dropout=MY_CONFIG["dropout"],
    batch_norm=MY_CONFIG["batch_norm"],
).to(device)

print(my_model)

my_model, my_history, my_best_acc, my_best_epoch = train_model(
    my_model,
    my_train_loader,
    my_val_loader,
    epochs=MY_CONFIG["epochs"],
    lr=MY_CONFIG["learning_rate"],
    weight_decay=MY_CONFIG["weight_decay"],
    optimizer_name=MY_CONFIG["optimizer"],
)

print()
print("=" * 60)
print(f"Baseline best validation accuracy : {baseline_best_acc:.4f}")
print(f"My best validation accuracy       : {my_best_acc:.4f}")
print(f"Best epoch                        : {my_best_epoch}")
print("=" * 60)


In [ ]:

plt.figure(figsize=(8, 4))
plt.plot(baseline_history["val_acc"], label="Baseline")
plt.plot(my_history["val_acc"], label="My Model")
plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy")
plt.title("Validation Accuracy Comparison")
plt.legend()
plt.show()



# 12. 여러 실험 결과 기록하기

좋은 실험에서는 **무엇을 바꾸었고 결과가 어떻게 달라졌는지 기록**하는 것이 중요합니다.

아래 표에 실험 결과를 직접 추가해 보세요.


In [ ]:

experiment_results = []

def record_experiment(name, config, best_val_acc, best_epoch):
    row = {
        "name": name,
        "hidden_dims": str(config["hidden_dims"]),
        "activation": config["activation"],
        "dropout": config["dropout"],
        "batch_norm": config["batch_norm"],
        "optimizer": config["optimizer"],
        "lr": config["learning_rate"],
        "weight_decay": config["weight_decay"],
        "batch_size": config["batch_size"],
        "epochs": config["epochs"],
        "best_epoch": best_epoch,
        "best_val_acc": best_val_acc,
    }
    experiment_results.append(row)

record_experiment(
    "My Model 1",
    MY_CONFIG,
    my_best_acc,
    my_best_epoch,
)

results_df = pd.DataFrame(experiment_results)
results_df.sort_values("best_val_acc", ascending=False)



## 선택 과제: 간단한 Grid Search

아래 코드는 여러 MLP 설정을 자동으로 비교하는 예시입니다.

실습 시간이 충분한 경우에만 실행하세요.

> 모든 조합을 무작정 탐색하기보다, 이전 실험 결과를 바탕으로 다음 실험을 설계하는 것을 권장합니다.


In [ ]:

# 실행 시간이 걸릴 수 있으므로 기본값은 False입니다.

RUN_SMALL_SEARCH = False

if RUN_SMALL_SEARCH:
    candidate_hidden_dims = [
        [128],
        [256, 128],
        [512, 256, 128],
    ]

    candidate_dropout = [0.0, 0.2]
    candidate_lr = [1e-3, 3e-4]

    search_results = []

    for hidden_dims in candidate_hidden_dims:
        for dropout in candidate_dropout:
            for lr in candidate_lr:

                model = MLP(
                    input_dim=INPUT_DIM,
                    hidden_dims=hidden_dims,
                    num_classes=NUM_CLASSES,
                    activation="relu",
                    dropout=dropout,
                    batch_norm=False,
                ).to(device)

                model, history, best_acc, best_epoch = train_model(
                    model,
                    train_loader,
                    val_loader,
                    epochs=20,
                    lr=lr,
                    weight_decay=0.0,
                    optimizer_name="adam",
                    verbose=False,
                )

                result = {
                    "hidden_dims": str(hidden_dims),
                    "dropout": dropout,
                    "lr": lr,
                    "best_epoch": best_epoch,
                    "best_val_acc": best_acc,
                }

                search_results.append(result)
                print(result)

    search_df = pd.DataFrame(search_results)
    display(search_df.sort_values("best_val_acc", ascending=False))



# 13. 최종 모델 선택

Validation 결과를 보고 가장 좋은 모델 구조와 hyperparameter를 결정합니다.

아래 `FINAL_CONFIG`에는 **validation 실험을 통해 선택한 최종 설정**을 입력하세요.

> Test 결과를 보고 다시 hyperparameter를 변경하면 안 됩니다.


In [ ]:

# ============================================================
# TODO: validation을 기준으로 선택한 최종 설정을 입력하세요.
# ============================================================

FINAL_CONFIG = MY_CONFIG.copy()

FINAL_CONFIG



## 14. 최종 Test 평가

최종 모델이 이미 `my_model`이라면 그대로 test set에서 평가할 수 있습니다.

아래에서는 `my_model`을 예시로 사용합니다.

**주의:** 실제 과제에서는 validation을 기준으로 최종 모델을 완전히 결정한 뒤 이 cell을 실행하세요.


In [ ]:

final_test_loader = make_loader(
    X_test_norm,
    y_test,
    batch_size=FINAL_CONFIG["batch_size"],
    shuffle=False,
)

test_loss, test_acc, test_preds, test_targets = evaluate(
    my_model,
    final_test_loader,
)

print(f"Test loss     : {test_loss:.4f}")
print(f"Test accuracy : {test_acc:.4f}")


In [ ]:

print(
    classification_report(
        test_targets,
        test_preds,
        target_names=CLASS_NAMES,
        digits=4,
    )
)


In [ ]:

cm = confusion_matrix(test_targets, test_preds)

plt.figure(figsize=(7, 6))
plt.imshow(cm)
plt.xticks(range(6), CLASS_NAMES, rotation=45, ha="right")
plt.yticks(range(6), CLASS_NAMES)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.colorbar()

for i in range(6):
    for j in range(6):
        plt.text(j, i, cm[i, j], ha="center", va="center")

plt.tight_layout()
plt.show()



# 15. 제출 내용

다음 내용을 정리하여 제출하세요.

### 1) Baseline
- Baseline validation accuracy

### 2) Hyperparameter search
최소 **3개 이상의 서로 다른 설정**을 비교하세요.

각 실험에 대해:
- hidden layer 구조
- activation
- dropout
- optimizer
- learning rate
- batch size
- validation accuracy

### 3) Best Model
- 최종 선택한 모델 구조
- 최종 선택 이유
- best validation accuracy
- test accuracy

### 4) 분석
다음 질문에 간단히 답하세요.

1. Hidden layer를 깊게 만들수록 항상 성능이 좋아졌는가?
2. Hidden dimension을 크게 만들수록 항상 성능이 좋아졌는가?
3. Dropout은 성능에 어떤 영향을 주었는가?
4. Learning rate 변화가 학습에 어떤 영향을 주었는가?
5. Training accuracy가 가장 높은 모델과 validation accuracy가 가장 높은 모델은 반드시 같은가?
6. Validation set이 필요한 이유는 무엇인가?
7. Test set을 hyperparameter tuning에 사용하면 왜 안 되는가?

---

## 핵심 메시지

이번 실습의 목표는 단순히 높은 accuracy를 얻는 것이 아닙니다.

**Training → Validation-based Model Selection → Final Test Evaluation**

이라는 머신러닝 실험의 기본적인 절차를 이해하는 것이 가장 중요합니다.
